# Pet Faces Classification

[AI for Beginners Curriculum](https://github.com/microsoft/ai-for-beginners) se Lab Assignment.

### Data Haasil Karna

Is assignment mein, hum ek relatively simple classification task par focus karenge - pet faces ki classification. Hum [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) use karenge, jisme dogs aur cats ki 37 alag-alag breeds ki images hain. Chalo data ko download aur visualize karne se shuru karte hain.

**Note:** Oxford-IIIT Pet Dataset mein poore pets ki images hain. Extract kiye gaye folder mein images breed ke hisaab se organized hongi.

In [ ]:
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
!tar xfz images.tar.gz
!rm images.tar.gz

Hum ek general function define karenge jo list se images ki series display karega:

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np

def display_images(l,titles=None,fontsize=12):
    n=len(l)
    fig,ax = plt.subplots(1,n)
    for i,im in enumerate(l):
        ax[i].imshow(im)
        ax[i].axis('off')
        if titles is not None:
            ax[i].set_title(titles[i],fontsize=fontsize)
    fig.set_size_inches(fig.get_size_inches()*n)
    plt.tight_layout()
    plt.show()

Ab chalo saari class subdirectories se guzarte hain aur har class ki pehli kuch images display karte hain:

In [ ]:
# Note: The Oxford-IIIT Pet Dataset extracts to a folder named 'images'
# Images are named by breed (e.g., 'Abyssinian_1.jpg')
# We need to organize them into breed-specific subdirectories
import os
from collections import defaultdict

# Organize images by breed
if not os.path.exists('petfaces'):
    os.makedirs('petfaces')
    for img_file in os.listdir('images'):
        if img_file.endswith(('.jpg', '.png')):
            # Extract breed name from filename (everything before the last underscore and number)
            breed = '_'.join(img_file.split('_')[:-1])
            breed_dir = os.path.join('petfaces', breed)
            if not os.path.exists(breed_dir):
                os.makedirs(breed_dir)
            os.rename(os.path.join('images', img_file), os.path.join(breed_dir, img_file))

for cls in os.listdir('petfaces'):
    print(cls)
    display_images([Image.open(os.path.join('petfaces',cls,x)) 
                    for x in os.listdir(os.path.join('petfaces',cls))[:10]])

Chalo apne dataset mein classes ki number bhi define karte hain:

In [ ]:
num_classes = len(os.listdir('petfaces'))
num_classes

## Deep Learning ke liye Dataset Taiyaar Karna

Apne neural network ki training shuru karne ke liye, hume saari images ko tensors mein convert karna hoga aur labels (class numbers) ke liye bhi tensors banane honge. Zyadatar neural network frameworks mein images ke saath kaam karne ke liye simple tools hote hain:
* Tensorflow mein, `tf.keras.preprocessing.image_dataset_from_directory` use karein
* PyTorch mein, `torchvision.datasets.ImageFolder` use karein

Jaisa ki aapne upar di gayi pictures mein dekha, sabka aspect ratio lagbhag square hai, isliye hume saari images ko square size mein resize karna hoga. Saath hi, hum images ko minibatches mein organize kar sakte hain.

In [ ]:
# CODE TO LOAD DATASET

Ab hume dataset ko train aur test parts mein split karna hoga:

In [ ]:
# CODE TO DO TRAIN/TEST SPLIT

Ab chalo apne dataset mein tensors ka size print karte hain. Agar aapne sab kuch sahi kiya hai, toh training elements ka size yeh hona chahiye:
 * Tensorflow ke liye `(batch_size,image_size,image_size,3)`, PyTorch ke liye `batch_size,3,image_size,image_size`
 * Labels ke liye `batch_size`

Labels mein classes ki number honi chahiye.

In [ ]:
# Print tensor sizes

In [ ]:
# Display the data

## Neural Network Define Karein

Image classification ke liye, aapko shayad kai layers waala ek convolutional neural network define karna chahiye. Dhyan dene waali baatein:
* Pyramid architecture ko dhyan mein rakhein, yaani jaise-jaise aap gehraai mein jaate hain, filters ki number badhni chahiye.
* Layers ke beech activation function (ReLU) aur max pooling ko na bhoolein.
* Final classifier hidden layers ke saath ya bina ho sakta hai, lekin output neurons ki number classes ki number ke barabar honi chahiye.

Ek important baat yeh hai ki aakhri layer par activation function aur loss function sahi hona chahiye:
* Tensorflow mein, aap activation ke roop mein `softmax` aur loss ke roop mein `sparse_categorical_crossentropy` use kar sakte hain. Sparse categorical cross-entropy aur non-sparse ke beech fark yeh hai ki pehla output ko class number ke roop mein expect karta hai, one-hot vector ke roop mein nahi.
* PyTorch mein, aap aakhri layer ko bina activation function ke rakh sakte hain aur `CrossEntropyLoss` loss function use kar sakte hain. Yeh function automatically softmax apply karta hai.

In [ ]:
# CODE TO DEFINE NEURAL NETWORK

## Neural Network ko Train Karein

Ab hum neural network ko train karne ke liye taiyaar hain. Training ke dauraan, please har epoch par train aur test data ki accuracy collect karein, aur phir accuracy ko plot karein taaki dekha ja sake ki kahin overfitting toh nahi ho rahi.

> Training ko tez karne ke liye, agar GPU available ho toh use karein. Jabki TensorFlow/Keras automatically GPU use karega, PyTorch mein aapko training ke dauraan model aur data dono ko `.to()` method se GPU par le jaana hoga, taaki GPU acceleration ka fayda uthaya ja sake.

In [ ]:
# TRAIN THE NETWORK

In [ ]:
# PLOT THE ACCURACY on train and validation dataset

Overfitting ke baare mein kya kaha ja sakta hai? Model ki accuracy sudharne ke liye kya kiya ja sakta hai?

## Optional: Top-3 Accuracy Calculate Karein

Is exercise mein, hum 35 classes ke saath classification kar rahe the, isliye hamara result - lagbhag 50% validation accuracy - kaafi accha hai. Standard ImageNet dataset mein aur bhi zyada classes hoti hain - 1000.

Aise cases mein yeh ensure karna mushkil hota hai ki model **hamesha** sahi class predict kare. Kuch cases mein do breeds ek-doosre se bahut milti-julti hoti hain, aur model bahut similar probabilities return karta hai (jaise, 0.45 aur 0.43). Agar hum standard accuracy measure karein, toh ise galat case maana jaayega, bhale hi model ne bahut chhoti galti ki ho. Is wajah se, hum aksar ek aur metric measure karte hain - model ki sabse probable teen predictions ke andar accuracy.

Hum us case ko accurate maante hain agar target label model ki top-3 predictions mein shaamil hai.

Test dataset par top-3 accuracy calculate karne ke liye, aapko manually dataset par jaana hoga, prediction paane ke liye neural network apply karna hoga, aur phir calculation karni hogi. Kuch suggestions:

* Tensorflow mein, `tf.nn.in_top_k` function use karein yeh dekhne ke liye ki `predictions` (model ka output) `targets` ke reference mein top-k mein hain ya nahi (`k=3` ko parameter ke roop mein pass karein). Yeh function boolean values ka ek tensor return karta hai, jise `tf.cast` use karke `int` mein convert kiya ja sakta hai, aur phir `tf.reduce_sum` use karke add kiya ja sakta hai.
* PyTorch mein, aap `torch.topk` function use karke highest probabilities waali classes ke indices nikaal sakte hain, aur phir dekh sakte hain ki sahi class unmein shaamil hai ya nahi. Zyada suggestions ke liye [yeh](https://gist.github.com/weiaicunzai/2a5ae6eac6712c70bde0630f3e76b77b) dekhein.

In [ ]:
# CALCULATE TOP-3 ACCURACY

## Optional: Cats vs. Dogs Classification Banayein

Hum yeh bhi dekhna chahte hain ki hamari binary cats vs. dogs classification ka performance usi dataset par kitna accurate hoga. Yeh karne ke liye, hume labels ko adjust karna hoga:

In [ ]:
# Define dataset that contains only two labels: 0 = cat, 1 = dog
# Hint: use class name prefix to figure out which one is which

In [ ]:
# Define neural network architecture and train

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->